In [37]:
import os
import shutil
import zipfile
import arcpy
from arcgis.gis import GIS

from config import PROJECT_FOLDER, CITY_NAME, AGOL_FOLDER_NAME

gdb_path = os.path.join(PROJECT_FOLDER, "UrbanHeatEquity.gdb")
output_folder = os.path.join(PROJECT_FOLDER, "outputs")
publish_folder = os.path.join(output_folder, "publish")

os.makedirs(publish_folder, exist_ok=True)

final_tracts = os.path.join(gdb_path, "heat_equity_tracts_final_clean")
city_boundary = os.path.join(gdb_path, "city_of_phoenix_boundary")

if not arcpy.Exists(final_tracts):
    raise RuntimeError("Final tract feature class does not exist.")

# Create publication file geodatabase

In [38]:
publish_gdb = os.path.join(publish_folder, "UrbanHeatEquity_Final.gdb")

if arcpy.Exists(publish_gdb):
    arcpy.management.Delete(publish_gdb)

arcpy.management.CreateFileGDB(publish_folder, "UrbanHeatEquity_Final.gdb")

final_publish_fc = os.path.join(publish_gdb, "heat_equity_tracts_final_clean")
boundary_publish_fc = os.path.join(publish_gdb, "city_of_phoenix_boundary")

arcpy.management.CopyFeatures(final_tracts, final_publish_fc)
arcpy.management.CopyFeatures(city_boundary, boundary_publish_fc)

print("Publication geodatabase created:", publish_gdb)

Publication geodatabase created: C:\Users\ava_gotthard\Documents\UrbanHeatEquity\outputs\publish\UrbanHeatEquity_Final.gdb


# Zip File Geodatabase, skipping lock files

In [39]:
zip_path = os.path.join(publish_folder, "UrbanHeatEquity_Final.gdb.zip")

if os.path.exists(zip_path):
    os.remove(zip_path)

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zipf:
    for root, dirs, files in os.walk(publish_gdb):
        for file in files:
            if ".lock" in file.lower():
                continue

            file_path = os.path.join(root, file)
            arcname = os.path.relpath(file_path, publish_folder)
            zipf.write(file_path, arcname)

print("Zipped File Geodatabase:", zip_path)

Zipped File Geodatabase: C:\Users\ava_gotthard\Documents\UrbanHeatEquity\outputs\publish\UrbanHeatEquity_Final.gdb.zip


# Connect to ArcGIS Online

In [40]:
gis = GIS("home")

print("Signed in as:", gis.users.me.username)

Signed in as: gotthardava


# Upload File Geodatabase without folder handling

In [47]:
item_properties = {
    "title": f"{CITY_NAME} Urban Heat Island and Tree Canopy Equity Analysis",
    "type": "File Geodatabase",
    "tags": "urban heat island, tree canopy, environmental justice, Landsat, ACS, Phoenix, GIS portfolio",
    "snippet": "Tract-level heat equity indicators for Phoenix.",
    "description": """
This portfolio project evaluates urban heat exposure and tree canopy equity for City of Phoenix census tracts.

The final feature layer includes mean land surface temperature, mean tree canopy percentage,
median household income, demographic indicators, geodesic area, and a heat equity priority score.
"""
}

uploaded_item = gis.content.add(
    item_properties=item_properties,
    data=zip_path
)

print("Uploaded item:", uploaded_item.title)

C:\Users\ava_gotthard\AppData\Local\ESRI\conda\envs\arcgispro-py3-clone\Lib\site-packages\IPython\core\interactiveshell.py:3579: DeprecatedWarning: add is deprecated as of 2.3.0 and has been removed in 3.0.0. Use `Folder.add()` instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


Uploaded item: Phoenix Urban Heat Island and Tree Canopy Equity Analysis


# Publish hosted feature layer

In [48]:
published_layer = uploaded_item.publish()

published_layer.update({
    "title": f"{CITY_NAME} Heat Equity Priority Tracts",
    "snippet": "Hosted feature layer showing tract-level heat equity indicators for Phoenix.",
    "tags": [
        "urban heat island",
        "tree canopy",
        "environmental justice",
        "Phoenix",
        "GIS portfolio"
    ],
    "description": item_properties["description"]
})

print("Published hosted feature layer:")
print(published_layer.url)

Published hosted feature layer:
https://services6.arcgis.com/SDdpEAs6WyhEBmTu/arcgis/rest/services/Phoenix_Urban_Heat_Island_and_Tree_Canopy_Equity_Analysis/FeatureServer


# Final QA before cleanup

In [49]:
print("Final deliverables:")
print("File Geodatabase ZIP:", zip_path)
print("Hosted Feature Layer:", published_layer.url)

final_count = int(arcpy.management.GetCount(final_tracts)[0])
print("Final tract count:", final_count)

Final deliverables:
File Geodatabase ZIP: C:\Users\ava_gotthard\Documents\UrbanHeatEquity\outputs\publish\UrbanHeatEquity_Final.gdb.zip
Hosted Feature Layer: https://services6.arcgis.com/SDdpEAs6WyhEBmTu/arcgis/rest/services/Phoenix_Urban_Heat_Island_and_Tree_Canopy_Equity_Analysis/FeatureServer
Final tract count: 386


# Optional cleanup: remove intermediate geodatabase items

In [50]:
items_to_delete = [
    "az_places_all",
    "state_tracts_all",
    "phoenix_tracts_landsat_crs",
    "city_boundary_landsat_crs",
    "tract_lst_stats",
    "tract_canopy_stats"
]

for item in items_to_delete:
    path = os.path.join(gdb_path, item)

    if arcpy.Exists(path):
        arcpy.management.Delete(path)
        print("Deleted:", item)
    else:
        print("Not found:", item)

print("Intermediate geodatabase cleanup complete.")

Deleted: az_places_all
Deleted: state_tracts_all
Deleted: phoenix_tracts_landsat_crs
Deleted: city_boundary_landsat_crs
Deleted: tract_lst_stats
Deleted: tract_canopy_stats
Intermediate geodatabase cleanup complete.


# Optional cleanup: remove large intermediate rasters after publishing
# Keep this commented out until you are fully done making maps/screenshots.

In [1]:
delete_large_intermediate_rasters = False

if delete_large_intermediate_rasters:

    processed_folder = os.path.join(PROJECT_FOLDER, "data", "processed")
    raw_landsat_folder = os.path.join(PROJECT_FOLDER, "data", "raw", "landsat")

    raster_paths = [
        os.path.join(processed_folder, "lst_celsius_mosaic.tif"),
        os.path.join(processed_folder, "lst_celsius_city_of_phoenix.tif"),
        os.path.join(processed_folder, "tree_canopy_city_of_phoenix.tif")
    ]

    for raster in raster_paths:
        if arcpy.Exists(raster):
            arcpy.management.Delete(raster)
            print("Deleted raster:", raster)
  # Optional: delete raw Landsat files
    # shutil.rmtree(raw_landsat_folder, ignore_errors=True)

    print("Large raster cleanup complete.")

else:
    print("Large raster cleanup skipped. Recommended until StoryMap/report is complete.")

Large raster cleanup skipped. Recommended until StoryMap/report is complete.
